# Panel data analysis with Python

In [ ]:
import numpy as np
import pandas as pd

from panelbox import PooledOLS					# Pooled model
from panelbox import RandomEffects				# Random-effect model
from panelbox import FixedEffects				# Fixed-effect model
from panelbox import FirstDifferenceEstimator	# First difference model

from panelbox.experiment import PanelExperiment	# Compare the results of multiple models

## Panel data preparation

Consider `Gasoline` dataset. We load it from a file as a DataFrame

In [6]:
df = pd.read_csv('./panels-plm/Gasoline.csv')
df.head()

,country,year,lgaspcar,lincomep,lrpmg,lcarpcap
0,AUSTRIA,1960,4.173244,-6.474277,-0.334548,-9.766840
1,AUSTRIA,1961,4.100989,-6.426006,-0.351328,-9.608622
2,AUSTRIA,1962,4.073177,-6.407308,-0.379518,-9.457257
3,AUSTRIA,1963,4.059509,-6.370679,-0.414251,-9.343155
4,AUSTRIA,1964,4.037689,-6.322247,-0.445335,-9.237739


## Models' fitting via formula

In [7]:
mod_pl = PooledOLS(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
res_pl = mod_pl.fit()
res_pl

PanelResults(model='Pooled OLS', nobs=342, k_params=4)

In [8]:
mod_re = RandomEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
res_re = mod_re.fit()
res_re

PanelResults(model='Random Effects (GLS)', nobs=342, k_params=4)

In [10]:
mod_fe = FixedEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
res_fe = mod_fe.fit()
res_fe

PanelResults(model='Fixed Effects', nobs=342, k_params=3)

In [11]:
mod_fd = FirstDifferenceEstimator(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
res_fd = mod_fd.fit()
res_fd

PanelResults(model='First Difference', nobs=324, k_params=3)

## Panel regressions with robust inferences

Let's fit basic models with Arellano-Bond covariance matrix estimator

we call `.fit(cov_type='clustered')` method for all models

We collect fitted models into a table (t-stats in brackets)

In [ ]:
mod_pl = PooledOLS(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_re = RandomEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_fe = FixedEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_fd = FirstDifferenceEstimator(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')

res_pl = mod_pl.fit(cov_type='clustered')
res_re = mod_re.fit(cov_type='clustered')
res_fe = mod_fe.fit(cov_type='clustered')
res_fd = mod_fd.fit(cov_type='clustered')

Let's fit basic models with Driscoll-Kraay covariance matrix estimator

we call `.fit(cov_type='kernel')` method for all models

We collect fitted models into a table (standard errors in brackets)

In [15]:
mod_pl = PooledOLS(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_re = RandomEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_fe = FixedEffects(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')
mod_fd = FirstDifferenceEstimator(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=df, entity_col='country', time_col='year')

res_pl = mod_pl.fit(cov_type='driscoll_kraay')
res_re = mod_re.fit(cov_type='driscoll_kraay')
res_fe = mod_fe.fit(cov_type='driscoll_kraay')
res_fd = mod_fd.fit(cov_type='driscoll_kraay')

# Panel Experiment

We fit a collection of models and get a fitted report

In [28]:
experiment = PanelExperiment(
	formula="lgaspcar~lincomep+lrpmg+lcarpcap",
	data=df,
	entity_col="country",
	time_col="year"
)

experiment.fit_all_models(model_types=['pooled', 're', 'fe'], names=['PL', 'RE', 'FE'], cov_type="clustered")

{'PL': PanelResults(model='Pooled OLS', nobs=342, k_params=4),
 'RE': PanelResults(model='Random Effects (GLS)', nobs=342, k_params=4),
 'FE': PanelResults(model='Fixed Effects', nobs=342, k_params=3)}

In [31]:
print(experiment.compare_models().summary())

MODEL COMPARISON SUMMARY

Models Compared: 3
Comparison Date: 2026-09-10 16:31:57

Comparison Metrics:
--------------------------------------------------------------------------------
Model                     R²         R² Adj     AIC          BIC         
--------------------------------------------------------------------------------
PL                        0.8549     0.8536     -92.99       -77.65      
RE                        0.7311     0.7287     118.11       133.45      
FE                        0.8396     0.8296     -672.67      -657.33     
--------------------------------------------------------------------------------

Best Models by Metric:
--------------------------------------------------------------------------------
  • Highest R²: PL
  • Lowest AIC: FE
  • Lowest BIC: FE

